# PAMTRA: riming dependent parameterization using SSRGA 
## Full-bin interface example with HALO-(AC)³ data

Here, we show how the riming dependet parameterization from Maherndl et al.(2023a, in revision) can be used in PAMTRA to simulate radar spectra of rimed ice particles. We use measured, binned particle size distributions. This is not only applicable to in situ measurements, but also to atmospheric models with full bin microphysics.

The script requires `haloac3_dataset_for_pamtra.nc` from the [example_data]("https://uni-koeln.sciebo.de/s/28700CuFssmin8q"). 

Start with importing the required libraries and setting up the Notebook to show plots inline.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

import pyPamtra

%matplotlib inline

Open the HALO-(AC)³ example data set (link). To keep file size small, the file contains only observations of a single cloud. See Mech et al. (2022, https://www.nature.com/articles/s41597-022-01900-7), Moser et al. (2023, https://acp.copernicus.org/preprints/acp-2023-44/) and Maherndl et al. (2023b, submitted) for details of the processing. 

In [ ]:
def info_sistema():
    import platform

    # platform.release() restituisce la versione del kernel (es. '5.15.0-60-generic' vs '5.15.133.1-microsoft-standard-WSL2')
    kernel_version = platform.release().lower()
    
    if "microsoft" in kernel_version or "wsl" in kernel_version:
        base = "/mnt/c/Data/pamtra_db"
        print("Sei su WSL (Ubuntu)")
        return base
    else:
        base = "/home/roversi/work/pamtra_db"
        print("Sei su Linux nativo")
        return base

base = info_sistema()
base

In [ ]:
# if sys.platform == "darwin":
#     base = os.path.expanduser("~/Data/APP_MZS")

# if sys.platform == "win32":
#     base = "C:/Data/pamtra_db"
    
# else:  # Linux / HPC (tintin)
#     base = "/home/roversi/work/pamtra_db"

data = xr.load_dataset(f"{base}/data/haloac3_dataset_for_pamtra.nc")
data

Extract the particle size distributions (psd_ice and psd_liquid for ice particles and supercooled droplets) from the netCDF file. 

In [ ]:
psd_i = np.ma.masked_invalid(data.psd_ice.values).filled(0)
psd_l = np.ma.masked_invalid(data.psd_liquid.values).filled(0)
nBins = np.shape(psd_i)[1]

Also, we save the average and bounding diameter of each size bin

In [ ]:
Dmean = data.bin_mid.values*1e-6 # bc values in microns
Dbound = np.append(data.bin_min.values[:,0], data.bin_max.values[-1, 0])
Dbound = Dbound*1e-6

In [ ]:
Dbound, Dmean

Next, we create an empty pyPamtra object

In [ ]:
pam = pyPamtra.pyPamtra()

Even though we use the full-bin interface, we need to add a standard hydrometeor description to create the required data structures. For consistency, the same data fields are required as for the regular PAMTRA hydrometeor interface. However, many parameters are not relevant when using the full-bin interface. 

In [ ]:
pam.df.addHydrometeor((
        "liquid",  # name 
        -99.,  # aspect ratio (NOT RELEVANT)
        1,  # liquid - ice flag
        -99.,  # density (NOT RELEVANT)
        -99.,  # mass size relation prefactor a (NOT RELEVANT)
        -99.,  # mass size relation exponent b (NOT RELEVANT)
        -99.,  # area size relation prefactor alpha (NOT RELEVANT)
        -99.,  # area size relation exponent beta (NOT RELEVANT)
        0,  # moment provided later (NOT RELEVANT)
        nBins,  # number of bins
        "fullBin",  # distribution name (NOT RELEVANT)
        -99.,  # distribution parameter 1 (NOT RELEVANT)
        -99.,  # distribution parameter 2 (NOT RELEVANT)
        -99.,  # distribution parameter 3 (NOT RELEVANT)
        -99.,  # distribution parameter 4 (NOT RELEVANT)
        -99.,  # minimum diameter (NOT RELEVANT)
        -99.,  # maximum diameter (NOT RELEVANT)
        'mie-sphere',  # scattering model
        'khvorostyanov01_drops',  # fall velocity relation
        0.0  # canting angle
    ))

In [ ]:
pam.df.addHydrometeor((
    "ice",  # name ## "liquid" -> "ice"
    -99.,  # aspect ratio (NOT RELEVANT)
    -1,  # liquid - ice flag
    -99.,  # density (NOT RELEVANT)
    -99.,  # mass size relation prefactor a (NOT RELEVANT)
    -99.,  # mass size relation exponent b (NOT RELEVANT)
    -99.,  # area size relation prefactor alpha (NOT RELEVANT)
    -99.,  # area size relation exponent beta (NOT RELEVANT)
    0,  # moment provided later (NOT RELEVANT)
    nBins,  # number of bins
    "fullBin",  # distribution name (NOT RELEVANT)
    -99.,  # distribution parameter 1 (NOT RELEVANT)
    -99.,  # distribution parameter 2 (NOT RELEVANT)
    -99.,  # distribution parameter 3 (NOT RELEVANT)
    -99.,  # distribution parameter 4 (NOT RELEVANT)
    -99.,  # minimum diameter (NOT RELEVANT)
    -99.,  # maximum diameter (NOT RELEVANT)
    'ss-rayleigh-gans',  # scattering model ## SSRGA
    'heymsfield10_particles',  # fall velocity relation ## more realistic for ice particles
    0.0  # canting angle
))

Next, we add an an atmospheric profile to PAMTRA. For this example, we use an US standard profile for simplicity. The `pyPamtra.importer.createUsStandardProfile` helper routine requires only height levels as an input. The height dimension of level properties is one longer than for layer properties such as hydrometeor properties. Therefore, we need to add one more height level to the ACME-V data set. Here, we simply add a height value to the vector. For comparison with e.g. a ground-based radar, the height levels should be derived from the height layers through interpolation. 

In [ ]:
pam = pyPamtra.importer.createUsStandardProfile(
    pam, 
    hgt_lev=list(data['alt6'].values) + [465]
)

Note that parameters not provided will be guessed, please make sure to look at the warning messages carefully. Now, the `pam.p` dictionary is created

In [ ]:
sorted(pam.p.keys())

The temperature and pressure fields have been populated with US standard atmosphere values in K and Pa, respectively. Note that all input quantities in PAMTRA except frequency (GHz) are in SI units. Refer to `pam.units` for details.

In [ ]:
pam.p['temp_lev'], pam.p['press_lev']

To model turbulence properly, we need to define the horizontal wind speed and the eddy dissipation rate (see Maahn et al. 2015, https://doi.org/10.1175/JTECH-D-14-00112.1 for details). For simplicity, we choose fixed values for this example

In [ ]:
pam.p['wind_uv'][:] = 10
pam.p['turb_edr'][:] = 1e-4

We set some non-default settings, see the documentation (https://pamtra.readthedocs.io/en/latest/settings.html) for details. Note the `pam.nmlSet["hydro_fullspec"] = True` which tells the Python Fortran kernel to use the provided distributions instead of the parameters of the hydrometeor description.

In [ ]:
pam.nmlSet["passive"] = False
pam.nmlSet["randomseed"] = 0
pam.nmlSet["radar_mode"] = "spectrum"
pam.nmlSet["radar_aliasing_nyquist_interv"] = 3
pam.nmlSet["hydro_adaptive_grid"] = False
pam.nmlSet["conserve_mass_rescale_dsd"] = False
pam.nmlSet["radar_use_hildebrand"] = True
pam.nmlSet["radar_noise_distance_factor"] = -2
pam.nmlSet["hydro_fullspec"] = True

For debugging, verbosity of the Fortran and Python code can be increased. Note that due to technical limitations (https://github.com/ipython/ipykernel/issues/110) the output of the Fortran kernel does not show up in Jupyter. For debugging the Fortran kernel, you can start `iPython` in a terminal and run this script with `%run fullbin-acmev-example.ipynb` to see the debugging output.

In [ ]:
pam.set["verbose"] = 0
pam.set["pyVerbose"] = 0

Finally, we create the Python objects for the measured DSDs

In [ ]:
pam.df.addFullSpectra()

which creates the `pam.df.dataFullSpec` dictionary containing empty arrays which need to be populated 

In [ ]:
list(pam.df.dataFullSpec.keys())

We start with adding the in situ observations for `d_bound_ds` (size bin boundaries in m), `d_ds` (size bin center in m, used for scattering calculation), and `n_ds` (number concentration in 1/m$^3$). Note that the in situ data set contains a drop size distribution in 1/m$^4$ which is why we apply `np.diff(Dbound)`.

In [ ]:
pam.df.dataFullSpec["d_bound_ds"][:] = Dbound
pam.df.dataFullSpec["d_ds"][:] = Dmean
pam.df.dataFullSpec["n_ds"][0,0,:,0,:] = (psd_l) * np.diff(Dbound)  
pam.df.dataFullSpec['n_ds'][0,0,:,1,:] = (psd_i) * np.diff(Dbound)

Note that the dimension of these arrays is

In [ ]:
pam.df.dataFullSpec["n_ds"].shape

which is for x-dimension, y-dimension, height, hydrometeor type (in case there are more than one), and size bin. Therefore, `np.newaxis` needs to be used for the measured DSDs to allow broadcasting to the required shape. 

It is crucial to define also the other hydrometeor properties `rho_ds` (particle density in kg/m$^3$), `area_ds` (cross section area in m$^2$), `mass_ds` (particle mass in kg), and `as_ratio` (aspect ratio, oblate for values < 1). However, for liquid cloud and drizzle drops, the trivial relations for spheres can be used. This give the opportunity to use arbitrarily complex relations for ice and snow particles. Note that it is the sole responsibility of the user to make sure all relations are consistent with each other, unless in PAMTRA's normal mode, the full-bin interface does not do any consistency checks. 

#### liquid droplets:
assumed to be spherical; we use Mie scattering

In [ ]:
# liquid
pam.df.dataFullSpec["rho_ds"][0,0,:,0,:] = 1000.
pam.df.dataFullSpec["area_ds"][0,0,:,0,:] = (np.pi / 4. * pam.df.dataFullSpec["d_ds"][0,0,:,0,:]**2)
pam.df.dataFullSpec["mass_ds"][0,0,:,0,:] = (np.pi / 6. * pam.df.dataFullSpec["rho_ds"][0,0,:,0,:] * 
                                             pam.df.dataFullSpec["d_ds"][0,0,:,0,:]**3)
pam.df.dataFullSpec["as_ratio"][0,0,:,0,:] = 1.0

#### rimed ice particles:

Now we add scattering and physical ice partile properties for our rimed ice particle population. We can provide the normalized rime mass M as a scalar or vector (different value for each time step). 

Normalized rime mass retrival values from Maherndl et al. (2023b, submitted):

In [ ]:
M = data.M.values

Alternatively: choose fixed M:      

In [ ]:
#M = np.full(data.time.size, 0.01) # assume normalized rime mass 

Mass (area) can be calculated from power law relations with prefactor a_m (a_a) and exponent b_m (b_a). ssrga_parameter(M) gives the SSRGA parameter kappa, beta, gamma, zeta_1 and alpha_eff as output. alpha_eff is equivalent to the aspect ratio here. 

In [ ]:
# rimed ice particles
a_m = np.expand_dims(pyPamtra.descriptorFile.riming_dependent_mass_size(M, 'dendrite')[0], 1) # mass size prefactor
b_m = np.expand_dims(pyPamtra.descriptorFile.riming_dependent_mass_size(M, 'dendrite')[1], 1) # mass size exponent
a_a = np.expand_dims(pyPamtra.descriptorFile.riming_dependent_area_size(M, 'dendrite')[0], 1) # area size prefactor 
b_a = np.expand_dims(pyPamtra.descriptorFile.riming_dependent_area_size(M, 'dendrite')[1], 1) # area size exponent 

pam.df.dataFullSpec["area_ds"][0,0,:,1,:] = (a_a * pam.df.dataFullSpec["d_ds"][0,0,:,1,:]**b_a)
pam.df.dataFullSpec["mass_ds"][0,0,:,1,:] = (a_m * pam.df.dataFullSpec["d_ds"][0,0,:,1,:]**b_m)

pam.df.dataFullSpec["as_ratio"][0,0,:,1,:]    = np.expand_dims(pyPamtra.descriptorFile.ssrga_parameter(M)[4], 1) # alpha_eff
pam.df.dataFullSpec["rg_kappa_ds"][0,0,:,1,:] = np.expand_dims(pyPamtra.descriptorFile.ssrga_parameter(M)[0], 1) 
pam.df.dataFullSpec["rg_beta_ds"][0,0,:,1,:]  = np.expand_dims(pyPamtra.descriptorFile.ssrga_parameter(M)[1], 1) 
pam.df.dataFullSpec["rg_gamma_ds"][0,0,:,1,:] = np.expand_dims(pyPamtra.descriptorFile.ssrga_parameter(M)[2], 1) 
pam.df.dataFullSpec["rg_zeta_ds"][0,0,:,1,:]  = np.expand_dims(pyPamtra.descriptorFile.ssrga_parameter(M)[3], 1)

Finally, we can run PAMTRA for the intended frequencies to estimate the radar observables. It is recommended to check whether `pam.fortError == 0` after running PAMTRA to catch errors in the Fortran part which are not displayed in Jupyter as discussed above. 

In [ ]:
frequencies = [94]
pam.runPamtra(frequencies)
print((pam.fortError))

Now, we can analyze the results which are stored in the `pam.r` dictionary.

In [ ]:
list(pam.r.keys())

For plotting, we extract the Doppler velocity bins `pam.r['radar_vel']`, the height layers `pam.p['hgt']`, the calculated radar Doppler spectrum `pam.r['radar_spectra']`, and radar reflectivity `pam.r['Ze']`.

In [ ]:
pam_velocity = pam.r['radar_vel'].squeeze()
pam_height = pam.p['hgt'].squeeze()
pam_spectra = pam.r['radar_spectra'].squeeze()
Ze = pam.r['Ze'].squeeze()

For comparison, we compare this to the measured radar reflectivity (with the 94 GHz MiRAC cloud radar. Her we use the attenuation corrected vlaues. Because the M data was derived based on a closure of measured reflectivities and in situ PSD, we expect Ze and data_Ze to fit well.

In [ ]:
data_Ze = data['Ze_mirac_corr'].values
data_height = data['alt6'].values
data_psd_i = np.log10(psd_i)

Create the plots:

In [ ]:
fig, [ax1, ax2] = plt.subplots(ncols=2, figsize=(10, 8), sharey=True)

pm = ax1.pcolormesh(Dmean, data_height, data_psd_i, rasterized=True)
plt.colorbar(pm, label='Ice particle concentration [m$^{-4}$]', ax=ax1)
#ax1.plot(data_deff, data_height, c='k', lw=3)

ax1.set_xscale('log')
ax1.set_title('%s UTC' % str(data.time[0].values).split('.')[0])
ax1.set_xlabel('Ice particle size [m]')
ax1.set_ylabel('Altitude [m]')
ax1.set_xlim(50e-6,Dmean.max())

ax1t = ax1.twiny()
ax1t.plot(data_Ze, data_height, c='k', lw=3)
ax1t.set_ylim(pam_height.min(), pam_height.max())
ax1t.set_xlabel('Radar reflectivity factor (black) $Z_e$ [dBz]\nmeasured by cloud radar MiRAC')

pm = ax2.pcolormesh(pam_velocity,
                    pam_height,
                    pam_spectra,
                    vmax=-5,
                    rasterized=True)
plt.colorbar(pm, label='Spectral reflectivity [dB]', ax=ax2)

ax2.set_xlim(-0.5, 5)
ax2.set_xlabel('Doppler velocity [m s$-1$]')

ax2t = ax2.twiny()
ax2t.plot(Ze, pam_height, c='w', lw=3)
ax2t.set_ylim(pam_height.min(), pam_height.max())
ax2t.set_xlabel('Radar reflectivity factor (black) $Z_e$ [dBz]\nforward simulated by PAMTRA')

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
plt.scatter(Ze, data_Ze)
ax.set_xlim(-5,5)
ax.set_ylim(-5,5)
ax.set_aspect(1)
ax.plot([-10,5], [-10,5], color='k', linestyle='-.')
ax.set_xlabel('PAMTRA (simulated) radar reflectivity [dBZ]')
ax.set_ylabel('MiRAC (measured) radar reflectivity [dBZ]')

root mean square error:

In [ ]:
rmse = np.sqrt(sum((data_Ze-Ze)**2)/Ze.size)
rmse

The results are not 1:1 the same due to: retrieval uncertanties and slightly different PAMTRA settings (e.g. standard atmosphere here instead of closest radio sonde profile). 